# Task 1

This section loads the NSW region summary dataset and prepares a cleaned version for analysis. The main issue in this file is that many indicators are only available in selected years, so missing values should not be filled with zero. Instead, the workflow keeps the original values, standardises the column names, extracts the unit from each indicator description, counts how many valid years each indicator has, and reshapes the dataset into a long format for later statistics and visualisation.

In [1]:
import pandas as pd
from pathlib import Path
from IPython.display import display

current_dir = Path.cwd()
data_path = current_dir / "data" / "Region summary_ New South Wales STE 1.csv"

df = pd.read_csv(data_path)
display(df.head())


,Measure Code,Parent Description,Description,2011,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,ERP_P_20,Estimated resident population - year ended 30 ...,Estimated resident population (no.),NaN,NaN,NaN,NaN,NaN,8046748.0,8110610.0,8097062.0,8166704.0,8341199.0,8479314.0,NaN
1,ERP_21,Estimated resident population - year ended 30 ...,Population density (persons/km2),NaN,NaN,NaN,NaN,NaN,10.0,10.1,10.1,10.2,10.4,10.6,NaN
2,ERP_M_20,Estimated resident population - year ended 30 ...,Estimated resident population - males (no.),NaN,NaN,NaN,NaN,NaN,3999452.0,4030710.0,4025393.0,4059763.0,4149032.0,4217861.0,NaN
3,ERP_F_20,Estimated resident population - year ended 30 ...,Estimated resident population - females (no.),NaN,NaN,NaN,NaN,NaN,4047296.0,4079900.0,4071669.0,4106941.0,4192167.0,4261453.0,NaN
4,ERP_19,Estimated resident population - year ended 30 ...,Median age - males (years),NaN,NaN,NaN,NaN,NaN,36.8,37.2,37.7,37.7,37.5,37.5,NaN


## Cleaning logic

The raw table is a wide table, which means each row is one indicator and each year is stored in a separate column. To make the data easier to analyse, the cleaning step below does three things. First, it standardises the text column names and year column names. Second, it separates the measurement unit from the description column and records how many non-null years each indicator has. Third, it reshapes the table into a long format so that each row represents one indicator-year value.

In [2]:
task1_df = df.rename(
    columns={
        "Measure Code": "measure_code",
        "Parent Description": "parent_description",
        "Description": "description",
    }
).copy()

task1_df.columns = [f"y_{col}" if str(col).isdigit() else col for col in task1_df.columns]
year_cols = [col for col in task1_df.columns if col.startswith("y_")]

task1_df["unit"] = task1_df["description"].str.extract(r"\(([^()]*)\)\s*$")
task1_df["description_clean"] = task1_df["description"].str.replace(r"\s*\([^()]*\)\s*$", "", regex=True)
task1_df["non_null_years"] = task1_df[year_cols].notna().sum(axis=1)
task1_df = task1_df[task1_df["non_null_years"] > 0].copy()

task1_long_df = task1_df.melt(
    id_vars=["measure_code", "parent_description", "description", "description_clean", "unit", "non_null_years"],
    value_vars=year_cols,
    var_name="year",
    value_name="value",
)

task1_long_df = task1_long_df.dropna(subset=["value"]).copy()
task1_long_df["year"] = task1_long_df["year"].str.replace("y_", "").astype(int)
task1_analysis_df = task1_long_df[task1_long_df["year"] != 2025].copy()

cleaning_summary_df = pd.DataFrame(
    {
        "item": [
            "original_rows",
            "rows_after_drop_all_null_years",
            "long_table_rows",
            "analysis_rows_without_2025",
            "rows_with_only_one_non_null_year",
            "non_null_values_in_2025",
        ],
        "value": [
            len(df),
            len(task1_df),
            len(task1_long_df),
            len(task1_analysis_df),
            int((task1_df["non_null_years"] == 1).sum()),
            int(task1_long_df[task1_long_df["year"] == 2025].shape[0]),
        ],
    }
)

display(cleaning_summary_df)
display(task1_df.head())
display(task1_analysis_df.head())


,item,value
0,original_rows,800
1,rows_after_drop_all_null_years,800
2,long_table_rows,2874
3,analysis_rows_without_2025,2871
4,rows_with_only_one_non_null_year,105
5,non_null_values_in_2025,3


,measure_code,parent_description,description,y_2011,y_2015,y_2016,y_2017,y_2018,y_2019,y_2020,y_2021,y_2022,y_2023,y_2024,y_2025,unit,description_clean,non_null_years
0,ERP_P_20,Estimated resident population - year ended 30 ...,Estimated resident population (no.),NaN,NaN,NaN,NaN,NaN,8046748.0,8110610.0,8097062.0,8166704.0,8341199.0,8479314.0,NaN,no.,Estimated resident population,6
1,ERP_21,Estimated resident population - year ended 30 ...,Population density (persons/km2),NaN,NaN,NaN,NaN,NaN,10.0,10.1,10.1,10.2,10.4,10.6,NaN,persons/km2,Population density,6
2,ERP_M_20,Estimated resident population - year ended 30 ...,Estimated resident population - males (no.),NaN,NaN,NaN,NaN,NaN,3999452.0,4030710.0,4025393.0,4059763.0,4149032.0,4217861.0,NaN,no.,Estimated resident population - males,6
3,ERP_F_20,Estimated resident population - year ended 30 ...,Estimated resident population - females (no.),NaN,NaN,NaN,NaN,NaN,4047296.0,4079900.0,4071669.0,4106941.0,4192167.0,4261453.0,NaN,no.,Estimated resident population - females,6
4,ERP_19,Estimated resident population - year ended 30 ...,Median age - males (years),NaN,NaN,NaN,NaN,NaN,36.8,37.2,37.7,37.7,37.5,37.5,NaN,years,Median age - males,6


,measure_code,parent_description,description,description_clean,unit,non_null_years,year,value
127,CENSUS_34,Aboriginal and Torres Strait Islander Peoples ...,Aboriginal and Torres Strait Islander Peoples ...,Aboriginal and Torres Strait Islander Peoples,no.,3,2011,172620.0
128,CENSUS_2,Aboriginal and Torres Strait Islander Peoples ...,Aboriginal and Torres Strait Islander Peoples (%),Aboriginal and Torres Strait Islander Peoples,%,3,2011,2.5
140,CENSUS_15,Religious affiliation - Census,Buddhism (%),Buddhism,%,3,2011,2.9
141,CENSUS_16,Religious affiliation - Census,Christianity (%),Christianity,%,3,2011,64.5
142,CENSUS_17,Religious affiliation - Census,Hinduism (%),Hinduism,%,3,2011,1.7


# Task 2

This section builds the geographic dataset for four selected areas. Each area is linked to one SA4 zone. For each selected SA4, the workflow first uses the NSW Administrative Boundaries API to find the SA2 regions contained inside that SA4. It then uses the NSW POI API to collect the POIs inside the bounding box of each SA2. Finally, the results are combined and saved to a local SQLite database.

## Selected SA4 codes

The group uses four SA4 codes: `120`, `117`, `118`, and `125`. The setup cell below matches each code to its official SA4 name and defines a small helper function so the same logic can be reused for each selected area.

In [3]:
import importlib
import task2_sa2

importlib.reload(task2_sa2)

available_sa4s = task2_sa2.list_nsw_sa4s()
area_zone_df = pd.DataFrame(
    {
        "area_name": ["area_1", "area_2", "area_3", "area_4"],
        "sa4_code": ["120", "117", "118", "125"],
    }
)

area_zone_df = area_zone_df.merge(
    available_sa4s.rename(columns={"SA4_CODE": "sa4_code", "SA4_NAME": "sa4_name"}),
    on="sa4_code",
    how="left",
)

def run_area_zone(area_name, sa4_code):
    sa2_bbox_df = task2_sa2.get_sa2_bbox_df_by_code(sa4_code)
    poi_df = task2_sa2.get_sa4_poi_df_by_code(sa4_code)
    sa2_bbox_df["area_name"] = area_name
    poi_df["area_name"] = area_name
    poi_count_df = poi_df.groupby("sa2_name").size().reset_index(name="poi_count")
    return sa2_bbox_df, poi_df, poi_count_df.sort_values("poi_count", ascending=False)

display(area_zone_df)


ModuleNotFoundError: No module named 'requests'

## Area 1

Area 1 is linked to `SA4_CODE = 120`, which corresponds to `Sydney - Inner West`. The cell below returns three tables: the SA2 regions contained in this SA4, a sample of the POI rows collected for this zone, and a count of how many POIs were assigned to each SA2.

In [ ]:
area_1_sa2_bbox_df, area_1_poi_df, area_1_poi_count_df = run_area_zone("area_1", "120")
display(area_1_sa2_bbox_df[["area_name", "sa4_code", "sa4_name", "sa2_main", "sa2_name"]])
display(area_1_poi_df.head())
display(area_1_poi_count_df)


,area_name,sa4_code,sa4_name,sa2_main,sa2_name
0,area_1,120,Sydney - Inner West,120011384,Concord West - North Strathfield
1,area_1,120,Sydney - Inner West,120021387,Balmain
2,area_1,120,Sydney - Inner West,120031390,Ashfield
3,area_1,120,Sydney - Inner West,120031393,Croydon Park - Enfield
4,area_1,120,Sydney - Inner West,120031396,Homebush
5,area_1,120,Sydney - Inner West,120031391,Burwood - Croydon
6,area_1,120,Sydney - Inner West,120031397,Strathfield
7,area_1,120,Sydney - Inner West,120011385,Drummoyne - Rodd Point
8,area_1,120,Sydney - Inner West,120021388,Leichhardt - Annandale
9,area_1,120,Sydney - Inner West,120031394,Dulwich Hill - Lewisham


,objectid,poiname,poitype,poigroup,longitude,latitude,sa4_code,sa4_name,sa2_main,sa2_name,area_name
0,848,CONCORD WEST POST OFFICE,Post Office,1,151.086920,-33.848207,120,Sydney - Inner West,120011384,Concord West - North Strathfield,area_1
1,852,None,Place Of Worship,1,151.100483,-33.861704,120,Sydney - Inner West,120011384,Concord West - North Strathfield,area_1
2,916,TREILLAGE TOWER,Observation Tower,3,151.079081,-33.847984,120,Sydney - Inner West,120011384,Concord West - North Strathfield,area_1
3,1120,CONCORD GOLF COURSE,Golf Course,3,151.097332,-33.850397,120,Sydney - Inner West,120011384,Concord West - North Strathfield,area_1
4,1128,HENLEY PARK,Park,3,151.098725,-33.859951,120,Sydney - Inner West,120011384,Concord West - North Strathfield,area_1


,sa2_name,poi_count
7,Drummoyne - Rodd Point,435
13,Lilyfield - Rozelle,370
12,Leichhardt - Annandale,352
4,Concord - Mortlake - Cabarita,294
1,Balmain,283
14,Strathfield,257
10,Haberfield - Summer Hill,197
0,Ashfield,193
9,Five Dock - Abbotsford,193
2,Burwood - Croydon,187


## Area 2

Area 2 is linked to `SA4_CODE = 117`, which corresponds to `Sydney - City and Inner South`. The same process is repeated here so that the results remain separate by area.

In [ ]:
area_2_sa2_bbox_df, area_2_poi_df, area_2_poi_count_df = run_area_zone("area_2", "117")
display(area_2_sa2_bbox_df[["area_name", "sa4_code", "sa4_name", "sa2_main", "sa2_name"]])
display(area_2_poi_df.head())
display(area_2_poi_count_df)


,area_name,sa4_code,sa4_name,sa2_main,sa2_name
0,area_2,117,Sydney - City and Inner South,117011324,Port Botany Industrial
1,area_2,117,Sydney - City and Inner South,117031338,Waterloo - Beaconsfield
2,area_2,117,Sydney - City and Inner South,117031335,Redfern - Chippendale
3,area_2,117,Sydney - City and Inner South,117021327,Petersham - Stanmore
4,area_2,117,Sydney - City and Inner South,117011321,Botany
5,area_2,117,Sydney - City and Inner South,117031330,Erskineville - Alexandria
6,area_2,117,Sydney - City and Inner South,117031336,Surry Hills
7,area_2,117,Sydney - City and Inner South,117011322,Mascot - Eastlakes
8,area_2,117,Sydney - City and Inner South,117031333,Potts Point - Woolloomooloo
9,area_2,117,Sydney - City and Inner South,117031334,Pyrmont - Ultimo


,objectid,poiname,poitype,poigroup,longitude,latitude,sa4_code,sa4_name,sa2_main,sa2_name,area_name
0,2791,FRED WILLIAMS RESERVE,Park,3,151.235054,-33.980692,117,Sydney - City and Inner South,117011324,Port Botany Industrial,area_2
1,3009,BOTANY MUNICIPAL GOLF LINKS,Golf Course,3,151.209774,-33.960649,117,Sydney - City and Inner South,117011324,Port Botany Industrial,area_2
2,3011,BICENTENNIAL PARK,Park,3,151.229946,-33.979046,117,Sydney - City and Inner South,117011324,Port Botany Industrial,area_2
3,3013,BOTANY CEMETERY,Cemetery,1,151.228252,-33.973462,117,Sydney - City and Inner South,117011324,Port Botany Industrial,area_2
4,3044,PRAISE EVANGELICAL FREE CHURCH OF AUSTRALIA,Place Of Worship,1,151.232717,-33.959938,117,Sydney - City and Inner South,117011324,Port Botany Industrial,area_2


,sa2_name,poi_count
16,Sydney - Haymarket - The Rocks,790
15,Sydenham - Tempe - St Peters,414
5,Marrickville,394
12,Pyrmont - Ultimo,263
11,Potts Point - Woolloomooloo,241
7,Newtown - Camperdown - Darlington,240
13,Redfern - Chippendale,157
4,Glebe - Forest Lodge,117
17,Sydney Airport,117
8,Pagewood - Hillsdale - Daceyville,113


## Area 3

Area 3 is linked to `SA4_CODE = 118`, which corresponds to `Sydney - Eastern Suburbs`. Keeping the outputs separate makes it easier to show which SA2 and POI records belong to each selected area.

In [ ]:
area_3_sa2_bbox_df, area_3_poi_df, area_3_poi_count_df = run_area_zone("area_3", "118")
display(area_3_sa2_bbox_df[["area_name", "sa4_code", "sa4_name", "sa2_main", "sa2_name"]])
display(area_3_poi_df.head())
display(area_3_poi_count_df)


,area_name,sa4_code,sa4_name,sa2_main,sa2_name
0,area_3,118,Sydney - Eastern Suburbs,118011344,Dover Heights
1,area_3,118,Sydney - Eastern Suburbs,118011341,Bondi Junction - Waverly
2,area_3,118,Sydney - Eastern Suburbs,118011347,Woollahra
3,area_3,118,Sydney - Eastern Suburbs,118021350,Malabar - La Perouse - Chifley
4,area_3,118,Sydney - Eastern Suburbs,118011342,Centennial Park
5,area_3,118,Sydney - Eastern Suburbs,118011339,Bondi - Tamarama - Bronte
6,area_3,118,Sydney - Eastern Suburbs,118021348,Coogee - Clovelly
7,area_3,118,Sydney - Eastern Suburbs,118021351,Maroubra
8,area_3,118,Sydney - Eastern Suburbs,118011345,Paddington - Moore Park
9,area_3,118,Sydney - Eastern Suburbs,118021352,Randwick


,objectid,poiname,poitype,poigroup,longitude,latitude,sa4_code,sa4_name,sa2_main,sa2_name,area_name
0,2299,RALEIGH RESERVE,Park,3,151.282547,-33.876566,118,Sydney - Eastern Suburbs,118011344,Dover Heights,area_3
1,2412,None,Park,3,151.283756,-33.868246,118,Sydney - Eastern Suburbs,118011344,Dover Heights,area_3
2,2424,DUMARESQ RESERVE,Park,3,151.270533,-33.866232,118,Sydney - Eastern Suburbs,118011344,Dover Heights,area_3
3,2473,ST ANDREW'S PRESBYTERIAN CHURCH,Place Of Worship,1,151.271366,-33.870861,118,Sydney - Eastern Suburbs,118011344,Dover Heights,area_3
4,2474,CAFFYN PARK,Park,3,151.278907,-33.872666,118,Sydney - Eastern Suburbs,118011344,Dover Heights,area_3


,sa2_name,poi_count
5,Double Bay - Bellevue Hill,286
12,Rose Bay - Vaucluse - Watsons Bay,232
8,Malabar - La Perouse - Chifley,226
10,Paddington - Moore Park,178
11,Randwick,172
9,Maroubra,155
7,Kensington - Kingsford,134
4,Coogee - Clovelly,126
2,Bondi Junction - Waverly,104
0,Bondi - Tamarama - Bronte,80


## Area 4

Area 4 is linked to `SA4_CODE = 125`, which corresponds to `Sydney - Parramatta`. This final section follows the same structure so that all four areas can be combined consistently later.

In [ ]:
area_4_sa2_bbox_df, area_4_poi_df, area_4_poi_count_df = run_area_zone("area_4", "125")
display(area_4_sa2_bbox_df[["area_name", "sa4_code", "sa4_name", "sa2_main", "sa2_name"]])
display(area_4_poi_df.head())
display(area_4_poi_count_df)


,area_name,sa4_code,sa4_name,sa2_main,sa2_name
0,area_4,125,Sydney - Parramatta,125031487,Yennora Industrial
1,area_4,125,Sydney - Parramatta,125031481,Granville - Clyde
2,area_4,125,Sydney - Parramatta,125041493,Toongabbie - Constitution Hill
3,area_4,125,Sydney - Parramatta,125011473,Homebush Bay - Silverwater
4,area_4,125,Sydney - Parramatta,125031484,Guildford West - Merrylands West
5,area_4,125,Sydney - Parramatta,125041490,North Rocks
6,area_4,125,Sydney - Parramatta,125021476,Carlingford
7,area_4,125,Sydney - Parramatta,125031485,Merrylands - Holroyd
8,area_4,125,Sydney - Parramatta,125031479,Chester Hill - Sefton
9,area_4,125,Sydney - Parramatta,125041488,Girraween - Westmead


,objectid,poiname,poitype,poigroup,longitude,latitude,sa4_code,sa4_name,sa2_main,sa2_name,area_name
0,1960,LINNWOOD MUSEUM,Museum,1,150.976257,-33.854700,125,Sydney - Parramatta,125031487,Yennora Industrial,area_4
1,1961,CARRINGTON ROAD PARK,Park,3,150.979534,-33.858424,125,Sydney - Parramatta,125031487,Yennora Industrial,area_4
2,2038,MCCREDIE PARK,Park,3,150.972060,-33.854984,125,Sydney - Parramatta,125031487,Yennora Industrial,area_4
3,2252,None,Place Of Worship,1,150.979047,-33.856015,125,Sydney - Parramatta,125031487,Yennora Industrial,area_4
4,2253,None,Place Of Worship,1,150.980362,-33.854497,125,Sydney - Parramatta,125031487,Yennora Industrial,area_4


,sa2_name,poi_count
17,Parramatta - Rosehill,385
5,Girraween - Westmead,308
11,Lidcombe - Regents Park,274
15,Northmead,254
10,Homebush Bay - Silverwater,240
12,Merrylands - Holroyd,232
13,North Parramatta,219
0,Auburn,210
6,Granville - Clyde,186
20,Toongabbie - Constitution Hill,181


## Combine and save

After each area's SA2 and POI data has been collected, the four outputs are concatenated into two full tables: one SA2 table and one POI table. These tables are then saved into a local SQLite database. The database links `poi` to `sa2_bbox` through the `sa2_main` field, which makes it possible to run joined queries later.

In [ ]:
all_sa2_bbox_df = pd.concat(
    [area_1_sa2_bbox_df, area_2_sa2_bbox_df, area_3_sa2_bbox_df, area_4_sa2_bbox_df],
    ignore_index=True,
)

all_poi_df = pd.concat(
    [area_1_poi_df, area_2_poi_df, area_3_poi_df, area_4_poi_df],
    ignore_index=True,
)

all_sa2_bbox_df = all_sa2_bbox_df[
    ["area_name", "sa4_code", "sa4_name", "sa2_main", "sa2_name", "state_name", "area_sqkm", "bbox_xmin", "bbox_ymin", "bbox_xmax", "bbox_ymax"]
]
all_poi_df = all_poi_df[
    ["objectid", "poiname", "poitype", "poigroup", "longitude", "latitude", "area_name", "sa4_code", "sa4_name", "sa2_main", "sa2_name"]
]

db_path = current_dir / "data" / "task2_poi.db"
task2_sa2.save_to_sqlite(all_sa2_bbox_df, all_poi_df, str(db_path))

schema_df = task2_sa2.read_sql(
    "select name, sql from sqlite_master where type = 'table' order by name",
    str(db_path),
)
join_df = task2_sa2.read_sql(
    "select p.poi_id, p.area_name, p.poiname, p.poitype, s.sa4_code, s.sa4_name, s.sa2_name from poi p join sa2_bbox s on p.sa2_main = s.sa2_main limit 20",
    str(db_path),
)

display(schema_df)
display(join_df)


,name,sql
0,poi,CREATE TABLE poi (\n poi_id integer...
1,sa2_bbox,CREATE TABLE sa2_bbox (\n area_name...
2,sqlite_sequence,"CREATE TABLE sqlite_sequence(name,seq)"


,poi_id,area_name,poiname,poitype,sa4_code,sa4_name,sa2_name
0,1,area_1,CONCORD WEST POST OFFICE,Post Office,120,Sydney - Inner West,Concord West - North Strathfield
1,2,area_1,None,Place Of Worship,120,Sydney - Inner West,Concord West - North Strathfield
2,3,area_1,TREILLAGE TOWER,Observation Tower,120,Sydney - Inner West,Concord West - North Strathfield
3,4,area_1,CONCORD GOLF COURSE,Golf Course,120,Sydney - Inner West,Concord West - North Strathfield
4,5,area_1,HENLEY PARK,Park,120,Sydney - Inner West,Concord West - North Strathfield
5,6,area_1,CENTRAL PARK,Park,120,Sydney - Inner West,Concord West - North Strathfield
6,7,area_1,WARBRICK PARK,Park,120,Sydney - Inner West,Concord West - North Strathfield
7,8,area_1,MUTTON RESERVE,Park,120,Sydney - Inner West,Concord West - North Strathfield
8,9,area_1,WALKER RESERVE,Park,120,Sydney - Inner West,Concord West - North Strathfield
9,10,area_1,BENNELONG PARK,Park,120,Sydney - Inner West,Concord West - North Strathfield
